# CMIP6 Model Evaluation & Ranking

Compares each CMIP6 GCM (extracted via the GEE notebook, same Date x Station format)
against IMD observed rainfall (`Representative_Stations_Rainfall.xlsx` from Tab 1),
computes performance metrics (R2, NSE, RMSE, MAE, PBIAS, KGE) per station and per
model, ranks the models, and produces comparison plots — all downloadable.

**Inputs**
1. `Representative_Stations_Rainfall.xlsx` (IMD observed, from Tab 1)
2. One or more `CMIP6_<model>_*.xlsx` files (from the GEE extraction notebook) —
   same Date x Station_ID columns

**Comparison scale**: monthly totals (more robust for GCM-vs-point comparison than
daily). Daily metrics are also computed for completeness.


## 1. Setup & upload files

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from google.colab import files

print("Select Representative_Stations_Rainfall.xlsx (IMD observed) ...")
imd_upload = files.upload()
imd_path = list(imd_upload.keys())[0]
imd_daily = pd.read_excel(imd_path, parse_dates=["Date"])
station_cols = [c for c in imd_daily.columns if c != "Date"]
print(f"IMD daily: {imd_daily.shape}, stations: {station_cols}")


In [ ]:
print("Select one or more CMIP6_<model>_*.xlsx files (multi-select allowed) ...")
cmip_upload = files.upload()

model_daily = {}
for fname in cmip_upload.keys():
    # filename pattern: CMIP6_<model>_<scenario>_<years>.xlsx
    model_name = fname.split("CMIP6_")[-1].split("_")[0] if "CMIP6_" in fname else fname.replace(".xlsx", "")
    df = pd.read_excel(fname, parse_dates=["Date"])
    model_daily[model_name] = df
    print(f"  {model_name}: {df.shape}")


## 2. Aggregate to monthly totals (Date -> Year/Month)

In [ ]:
def to_monthly(df, station_cols):
    d = df.copy()
    d["Year"] = d["Date"].dt.year
    d["Month"] = d["Date"].dt.month
    monthly = d.drop(columns="Date").groupby(["Year", "Month"])[station_cols].sum(min_count=1).reset_index()
    return monthly


imd_monthly = to_monthly(imd_daily, station_cols)

model_monthly = {}
for model, df in model_daily.items():
    cols = [c for c in df.columns if c != "Date"]
    model_monthly[model] = to_monthly(df, cols)

print(f"IMD monthly: {imd_monthly.shape}")
for m, df in model_monthly.items():
    print(f"{m} monthly: {df.shape}")


## 3. Metric functions

In [ ]:
def r2_score_(obs, sim):
    return np.corrcoef(obs, sim)[0, 1] ** 2


def nse(obs, sim):
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - obs.mean()) ** 2)


def rmse(obs, sim):
    return np.sqrt(np.mean((obs - sim) ** 2))


def mae(obs, sim):
    return np.mean(np.abs(obs - sim))


def pbias(obs, sim):
    return 100 * np.sum(sim - obs) / np.sum(obs)


def kge(obs, sim):
    r = np.corrcoef(obs, sim)[0, 1]
    alpha = sim.std() / obs.std()
    beta = sim.mean() / obs.mean()
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


def compute_metrics(obs, sim):
    mask = (~np.isnan(obs)) & (~np.isnan(sim))
    obs, sim = obs[mask], sim[mask]
    return {
        "R2": r2_score_(obs, sim),
        "NSE": nse(obs, sim),
        "RMSE": rmse(obs, sim),
        "MAE": mae(obs, sim),
        "PBIAS": pbias(obs, sim),
        "KGE": kge(obs, sim),
        "n": len(obs),
    }


## 4. Compute metrics per model per station (monthly)

In [ ]:
results = []
for model, mdf in model_monthly.items():
    merged = imd_monthly.merge(mdf, on=["Year", "Month"], suffixes=("_obs", "_sim"))
    for st in station_cols:
        if st not in mdf.columns:
            continue
        obs = merged[f"{st}_obs"].values.astype(float)
        sim = merged[f"{st}_sim"].values.astype(float)
        m = compute_metrics(obs, sim)
        m["Model"] = model
        m["Station"] = st
        results.append(m)

metrics_df = pd.DataFrame(results)
metrics_df = metrics_df[["Model", "Station", "R2", "NSE", "RMSE", "MAE", "PBIAS", "KGE", "n"]]
metrics_df


## 5. Model ranking (basin-average across stations)

In [ ]:
ranking = (metrics_df.groupby("Model")[["R2", "NSE", "RMSE", "MAE", "PBIAS", "KGE"]]
           .mean()
           .reset_index())

# Higher NSE/R2/KGE = better; lower RMSE/MAE/|PBIAS| = better
ranking["Rank_NSE"] = ranking["NSE"].rank(ascending=False)
ranking["Rank_KGE"] = ranking["KGE"].rank(ascending=False)
ranking["Rank_RMSE"] = ranking["RMSE"].rank(ascending=True)
ranking["Overall_Rank_Score"] = ranking[["Rank_NSE", "Rank_KGE", "Rank_RMSE"]].mean(axis=1)
ranking = ranking.sort_values("Overall_Rank_Score").reset_index(drop=True)

ranking


## 6. Plots — scatter (IMD vs each model, basin total), ranking bar chart, Taylor diagram

In [ ]:
# Basin-average monthly series (mean across stations)
imd_basin = imd_monthly[station_cols].mean(axis=1)

fig_scatter, axes = plt.subplots(1, len(model_monthly), figsize=(5 * len(model_monthly), 5), squeeze=False)
for ax, (model, mdf) in zip(axes[0], model_monthly.items()):
    common_cols = [c for c in station_cols if c in mdf.columns]
    sim_basin = mdf[common_cols].mean(axis=1)
    n = min(len(imd_basin), len(sim_basin))
    x, y = imd_basin.values[:n], sim_basin.values[:n]
    ax.scatter(x, y, alpha=0.5, s=15)
    lims = [0, max(x.max(), y.max())]
    ax.plot(lims, lims, "k--", linewidth=1, label="1:1")
    r2 = r2_score_(x, y)
    ax.set_title(f"{model} (R2={r2:.2f})")
    ax.set_xlabel("IMD monthly rainfall (mm)")
    ax.set_ylabel("Model monthly rainfall (mm)")
    ax.legend()
fig_scatter.tight_layout()
plt.show()


In [ ]:
fig_rank, ax_rank = plt.subplots(figsize=(7, 4))
ax_rank.bar(ranking["Model"], ranking["NSE"], color="#4a90d9")
ax_rank.set_ylabel("Mean NSE (across stations)")
ax_rank.set_title("Model ranking by NSE")
ax_rank.tick_params(axis="x", rotation=30)
fig_rank.tight_layout()
plt.show()


In [ ]:
# Simple Taylor-diagram-style plot: standard deviation ratio vs correlation
fig_taylor, ax_t = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})

obs_std = imd_basin.std()
for model, mdf in model_monthly.items():
    common_cols = [c for c in station_cols if c in mdf.columns]
    sim_basin = mdf[common_cols].mean(axis=1)
    n = min(len(imd_basin), len(sim_basin))
    x, y = imd_basin.values[:n], sim_basin.values[:n]
    r = np.corrcoef(x, y)[0, 1]
    std_ratio = y.std() / x.std()
    theta = np.arccos(r)
    ax_t.plot(theta, std_ratio, "o", label=model, markersize=8)

ax_t.plot(0, 1, "k*", markersize=15, label="IMD (reference)")
ax_t.set_thetamin(0)
ax_t.set_thetamax(90)
ax_t.set_xlabel("")
ax_t.set_title("Taylor diagram (correlation vs std-dev ratio)")
ax_t.legend(bbox_to_anchor=(1.3, 1.0))
plt.show()


## 7. Download results

In [ ]:
metrics_df.to_excel("/content/Model_Metrics_PerStation.xlsx", index=False)
ranking.to_excel("/content/Model_Ranking.xlsx", index=False)

fig_scatter.savefig("/content/scatter_plots.png", dpi=150, bbox_inches="tight")
fig_rank.savefig("/content/ranking_chart.png", dpi=150, bbox_inches="tight")
fig_taylor.savefig("/content/taylor_diagram.png", dpi=150, bbox_inches="tight")

for f in ["Model_Metrics_PerStation.xlsx", "Model_Ranking.xlsx",
          "scatter_plots.png", "ranking_chart.png", "taylor_diagram.png"]:
    files.download(f"/content/{f}")
